# StreamSense — Exploratory Data Analysis & Data Understanding

**Dataset:** MovieLens `ml-latest-small` (GroupLens Research) — a public benchmark of movie
ratings used here as a proxy for OTT viewing-engagement logs.

This notebook is the **first step** of the project: before training any model we characterise the
data and let its structure drive the architecture. Each section ends with a short *Design
implication* that motivates a concrete choice in the downstream pipeline (retrieval, ranking,
segmentation, content features, periodic retraining).

Runs on a **plain Colab runtime** — only pandas / numpy / matplotlib / seaborn / scikit-learn
(all pre-installed). No TFX or GPU required.

## 1. Setup and data download

In [ ]:
import io, zipfile, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 4.5)

URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
with urllib.request.urlopen(URL) as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
ROOT = "ml-latest-small"
print("Downloaded and extracted:", ROOT)

## 2. Load the tables

In [ ]:
ratings = pd.read_csv(f"{ROOT}/ratings.csv")   # userId, movieId, rating, timestamp
movies  = pd.read_csv(f"{ROOT}/movies.csv")    # movieId, title, genres
tags    = pd.read_csv(f"{ROOT}/tags.csv")      # userId, movieId, tag, timestamp
ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")
display(ratings.head())
display(movies.head())

## 3. Dataset at a glance

In [ ]:
n_users  = ratings.userId.nunique()
n_movies_rated = ratings.movieId.nunique()
n_movies_cat   = movies.movieId.nunique()
density = 100 * len(ratings) / (n_users * n_movies_cat)

summary = pd.DataFrame({
    "metric": ["# ratings (interactions)", "# users", "# movies in catalogue",
               "# movies actually rated", "ratings per user (min)",
               "rating scale", "date range", "user-item matrix density"],
    "value": [f"{len(ratings):,}", f"{n_users:,}", f"{n_movies_cat:,}",
              f"{n_movies_rated:,}", int(ratings.groupby('userId').size().min()),
              f"{ratings.rating.min()} – {ratings.rating.max()} (0.5 steps)",
              f"{ratings.date.min().date()} → {ratings.date.max().date()}",
              f"{density:.2f}%  (i.e. {100-density:.2f}% empty)"],
})
display(summary)

**Design implication.** The catalogue has ~9.7k items but the user–item matrix is **~98%
empty**. A model cannot score every item for every request, and pure memorisation (co-occurrence
counts) is too sparse to generalise. This is the core reason the system is **two-stage** (a cheap
retrieval model to shortlist, then an expensive ranker on the shortlist) and **embedding-based**
(dense vectors that generalise across the sparsity).

## 4. Rating distribution → defining the engagement label

In [ ]:
ax = (ratings.rating.value_counts().sort_index()
      .plot(kind="bar", color=sns.color_palette("deep")[0], edgecolor="white"))
ax.set(title="Distribution of rating values", xlabel="rating", ylabel="count")
ax.axvline(x=6.5, color="crimson", linestyle="--")   # boundary between <4 and >=4 on the 0.5-grid
plt.tight_layout(); plt.show()

pos = (ratings.rating >= 4.0).mean()
print(f"Mean rating: {ratings.rating.mean():.2f}")
print(f"Share of ratings >= 4.0 (our positive-engagement label): {pos*100:.1f}%")

**Design implication.** Ratings are skewed toward the high end. We binarise engagement as
**label = 1 if rating ≥ 4.0** — a clean, business-meaningful target ("the user liked it") for the
ranking model, and a reasonable near-balanced split for `binary_crossentropy` training.

## 5. Activity per user → heterogeneity motivates segmentation

In [ ]:
upc = ratings.groupby("userId").size()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(upc, bins=40, color=sns.color_palette("deep")[2], edgecolor="white")
ax[0].set(title="Ratings per user", xlabel="# ratings", ylabel="# users")
ax[1].plot(np.sort(upc.values), np.linspace(0,1,len(upc)))
ax[1].set(title="Cumulative distribution (ECDF)", xlabel="# ratings", ylabel="fraction of users")
plt.tight_layout(); plt.show()
print(f"ratings per user  — min {upc.min()}, median {int(upc.median())}, max {upc.max()}")

**Design implication.** User activity spans two orders of magnitude (from the 20-rating
minimum to power users with hundreds). A single global model under-serves both ends. We compute a
**user `segment`** (K-Means, Section 10) and feed it as a feature so the model can condition on
behavioural cohort — a light, production-friendly form of personalisation that also helps the
cold-ish, low-activity users.

## 6. Item popularity → the long tail that forces a retrieval stage

In [ ]:
mpc = ratings.groupby("movieId").size().sort_values(ascending=False)
cum = mpc.cumsum() / mpc.sum()
share_top20 = cum.iloc[int(0.2*len(cum))] * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(1, len(mpc)+1), mpc.values)
ax[0].set(title="Popularity (rank–frequency)", xlabel="movie rank", ylabel="# ratings",
          xscale="log", yscale="log")
ax[1].plot(np.linspace(0,100,len(cum)), cum.values*100)
ax[1].set(title="Cumulative share of interactions", xlabel="top X% of movies", ylabel="% of all ratings")
ax[1].axhline(share_top20, color="crimson", ls="--")
plt.tight_layout(); plt.show()
print(f"The top 20% most-popular movies account for {share_top20:.0f}% of all interactions.")
print(f"{(mpc<=5).mean()*100:.0f}% of movies have <=5 ratings (cold, long-tail items).")

**Design implication.** Classic long-tail: a small head of popular titles dominates
interactions while most of the catalogue is rarely watched. Two consequences: (1) a **popularity
baseline is a strong, honest control** for A/B evaluation, and (2) at serving time we can't rank the
whole catalogue per request — a **retrieval / candidate-generation stage** (two-tower) must first
shortlist a few hundred plausible items, which the ranker then orders. This is the single biggest
reason the architecture has two model layers instead of one.

## 7. Sparsity, visualised

In [ ]:
top_u = upc.sort_values(ascending=False).head(40).index
top_m = mpc.head(60).index
sub = (ratings[ratings.userId.isin(top_u) & ratings.movieId.isin(top_m)]
       .pivot_table(index="userId", columns="movieId", values="rating"))
sub = sub.reindex(index=top_u, columns=top_m)
plt.figure(figsize=(11,5))
sns.heatmap(sub.notna(), cbar=False, cmap="Blues")
plt.title("Even among the 40 most-active users × 60 most-popular movies, most cells are empty")
plt.xlabel("movie"); plt.ylabel("user"); plt.tight_layout(); plt.show()

**Design implication.** Sparsity persists even in the densest corner of the data. Dense
**embeddings** (learned user and item vectors) are required so the model can generalise to
user–item pairs it has never co-observed — which is exactly what the two-tower and the ranker's
embedding layers provide.

## 8. Temporal trend → why the pipeline is built for periodic retraining

In [ ]:
per_year = ratings.set_index("date").resample("YE").size()
per_year.plot(marker="o")
plt.title("Interactions per year"); plt.xlabel("year"); plt.ylabel("# ratings")
plt.tight_layout(); plt.show()

**Design implication.** Engagement volume and catalogue composition drift over the years.
A one-off trained model goes stale, so the training path is packaged as a **reproducible TFX
pipeline** compiled to a **Kubeflow/Vertex** spec that can be re-run on a schedule against fresh
data — the model is retrained and re-pushed, not hand-built.

## 9. Content signal → why we add GenAI content embeddings

In [ ]:
genre_long = movies.assign(genres=movies.genres.str.split("|")).explode("genres")
genre_long = genre_long[genre_long.genres != "(no genres listed)"]
gc = genre_long.genres.value_counts()

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
gc.sort_values().plot(kind="barh", ax=ax[0], color=sns.color_palette("deep")[3])
ax[0].set(title="Catalogue composition by genre", xlabel="# movies")

rg = ratings.merge(genre_long[["movieId","genres"]], on="movieId")
rg.groupby("genres").rating.mean().sort_values().plot(kind="barh", ax=ax[1],
    color=sns.color_palette("deep")[4])
ax[1].set(title="Mean rating by genre", xlabel="avg rating"); ax[1].set_xlim(3, 4.2)
plt.tight_layout(); plt.show()

**Design implication.** Genre (and title text) carry real preference signal and mean-rating
differences. Rather than only one-hot genres, we embed each title's *"title + genres"* text with a
**sentence-transformer** (a GenAI text encoder) to get a dense content vector. This adds a
content-based signal on top of collaborative signal — valuable for new/cold items that have little
interaction history, and the hook for a downstream "explain this recommendation" feature.

## 10. User segmentation preview (the `segment` feature)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# user genre-affinity profile: mean rating each user gives per genre
profile = (rg.pivot_table(index="userId", columns="genres", values="rating", aggfunc="mean")
             .fillna(0))
seg = KMeans(n_clusters=8, n_init="auto", random_state=0).fit_predict(profile.values)
pcs = PCA(n_components=2, random_state=0).fit_transform(profile.values)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sc = ax[0].scatter(pcs[:,0], pcs[:,1], c=seg, cmap="tab10", s=18)
ax[0].set(title="Users projected to 2D (PCA), coloured by K-Means segment",
          xlabel="PC1", ylabel="PC2")
pd.Series(seg).value_counts().sort_index().plot(kind="bar", ax=ax[1],
    color=sns.color_palette("deep")[0])
ax[1].set(title="Segment sizes", xlabel="segment", ylabel="# users")
plt.tight_layout(); plt.show()
print("Segment sizes:", np.bincount(seg))

**Design implication.** Users fall into distinct taste cohorts. In `features.py` this exact
K-Means (on content-embedding affinity) produces an **8-way `segment` label** that becomes a model
feature and an axis for slicing offline metrics. It is a cheap, interpretable personalisation signal
that complements the per-user embedding.

## 11. From EDA to the training data contract

The findings above define exactly what `features.py` emits and what the TFX pipeline
consumes. Each interaction row is written as a `tf.Example` with this schema:

| field | type | source | role |
|---|---|---|---|
| `user_id` | string | ratings | retrieval query + ranker embedding key |
| `movie_id` | string | ratings | retrieval candidate + ranker embedding key |
| `movie_title` | string | movies | content / retrieval candidate text |
| `segment` | int64 | K-Means (Sec. 10) | cohort feature |
| `rating` | float | ratings | raw signal |
| `label` | int64 | `rating >= 4.0` (Sec. 4) | ranking target |

The pipeline then splits these interactions **80/20 (train/eval by hashing)**, computes vocabularies
and transforms inside TFX `Transform`, and trains the ranker — everything downstream is a direct,
data-driven consequence of this EDA.

## 12. Key takeaways → architecture, in one table

| EDA finding | Architecture decision |
|---|---|
| ~98% sparse user–item matrix, 9.7k items | **Two-stage** design: retrieval shortlists, ranker orders |
| Heavy long-tail popularity | Can't score full catalogue per request → **two-tower retrieval**; popularity is the A/B baseline |
| Sparsity persists even among heavy users/popular items | **Dense embeddings** (two-tower + ranker) to generalise |
| Wide per-user activity range | **User segmentation** (`segment` feature) for cohort-level personalisation |
| Genre / text preference signal | **GenAI content embeddings** of title+genres as content features |
| Multi-year engagement drift | **TFX → Kubeflow/Vertex** reproducible, re-runnable training pipeline |
| High-rating skew | Binary **engagement label** at rating ≥ 4.0 for the ranker |

Next: `notebooks/` training (two-tower retrieval, TFX ranking pipeline) and serving
(TF Serving / TorchServe / Triton). See the project **README** for the full walkthrough.